# Random forest and logistic regression

Five stratified 10-fold runs per model on the original feature matrix. Requires the 2,448-row training matrix and aligned phenotype labels. Set PROJECT_ROOT to a working copy of the prepared study inputs. This notebook writes results under that directory.

In [ ]:
# ================================================================
# SECTION 18A — RANDOM FOREST + LOGISTIC REGRESSION
# FIVE-SEED, 10-FOLD ROBUSTNESS ANALYSIS
#
# Recommended configuration:
#   Google Colab CPU, preferably High-RAM
#
# Expected runtime:
#   Approximately 20–60 minutes
#
# Evaluation:
#   Seeds: 42, 123, 456, 789, 1000
#   10 stratified folds per seed
#   50 held-out evaluations per model
#
# Dataset:
#   N = 2,448 genomes
#   Features = 5,606
#   Susceptible = 1,815
#   Resistant = 633
#
# Important:
#   - No hyperparameter tuning is performed.
#   - Classification threshold remains 0.5.
#   - The existing master table is not modified.
# ================================================================

In [ ]:
# ================================================================
# 0. MOUNT DRIVE AND IMPORT PACKAGES
# ================================================================



import json
import time
import shutil
import warnings
import platform
import hashlib

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy
import sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.exceptions import ConvergenceWarning

import matplotlib.pyplot as plt

In [ ]:
# ================================================================
# 1. PATHS AND CONSTANTS
# ================================================================

PROJECT_ROOT = Path(
    "project_data"
)

REVISION_ROOT = (
    PROJECT_ROOT
    / "Frontiers_Revision_Outputs"
)

X_PATH = (
    REVISION_ROOT
    / "tables"
    / "X_bag_of_genes_binary.csv"
)

Y_PATH = (
    PROJECT_ROOT
    / "Final_Metrics"
    / "rf_y_true.csv"
)

BASE_OUT_DIR = (
    REVISION_ROOT
    / "tables"
    / "rf_lr_seed_robustness"
)

BASE_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_TAG = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RUN_DIR = (
    BASE_OUT_DIR
    / f"run_{RUN_TAG}"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

SEEDS = [
    42,
    123,
    456,
    789,
    1000,
]

N_SPLITS = 10
THRESHOLD = 0.5

EXPECTED_N = 2448
EXPECTED_FEATURES = 5606
EXPECTED_SUSCEPTIBLE = 1815
EXPECTED_RESISTANT = 633

In [ ]:
# ================================================================
# 2. BASIC FILE CHECKS
# ================================================================

if not X_PATH.exists():
    raise FileNotFoundError(
        f"Feature matrix was not found:\n{X_PATH}"
    )

if not Y_PATH.exists():
    raise FileNotFoundError(
        f"Label file was not found:\n{Y_PATH}"
    )

print("Feature-matrix file:")
print(X_PATH)

print("\nLabel file:")
print(Y_PATH)

In [ ]:
# ================================================================
# 3. OPTIONAL FILE HASHING FOR REPRODUCIBILITY
# ================================================================

def calculate_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


print("\nCalculating source-file hashes...")

X_SHA256 = calculate_sha256(X_PATH)
Y_SHA256 = calculate_sha256(Y_PATH)

print(f"X SHA256: {X_SHA256}")
print(f"y SHA256: {Y_SHA256}")

In [ ]:
# ================================================================
# 4. LOAD AND VALIDATE FEATURE MATRIX
# ================================================================

def load_feature_matrix(path):
    """
    Load the saved bag-of-genes matrix.

    Expected raw form:
        2,448 rows
        5,607 columns

    The additional column should be a genome identifier or a saved
    CSV index. It is removed before model training.
    """

    df = pd.read_csv(path)

    print("\n" + "=" * 72)
    print("RAW FEATURE-MATRIX INSPECTION")
    print("=" * 72)

    print(f"Raw X shape: {df.shape}")
    print(
        "First five column names:",
        df.columns[:5].tolist(),
    )

    sample_ids = None
    removed_column = None

    id_names = {
        "genome_id",
        "genome",
        "genome id",
        "sample_id",
        "sample",
        "sample id",
        "isolate_id",
        "isolate",
        "isolate id",
        "id",
        "index",
        "row_index",
        "row index",
    }

    target_names = {
        "label",
        "labels",
        "target",
        "y",
        "phenotype",
        "resistance",
        "resistance_status",
        "resistance status",
        "true_label",
        "true_labels",
    }

    # ------------------------------------------------------------
    # Case 1: Matrix already has exactly 5,606 feature columns.
    # ------------------------------------------------------------

    if df.shape[1] == EXPECTED_FEATURES:
        print(
            "The CSV already contains exactly 5,606 columns. "
            "No extra column was removed."
        )

    # ------------------------------------------------------------
    # Case 2: Matrix has 5,607 columns.
    # Identify and remove exactly one non-feature column.
    # ------------------------------------------------------------

    elif df.shape[1] == EXPECTED_FEATURES + 1:

        normalized_names = {
            col: str(col).strip().lower()
            for col in df.columns
        }

        unnamed_columns = [
            col
            for col in df.columns
            if normalized_names[col].startswith("unnamed")
        ]

        named_id_columns = [
            col
            for col in df.columns
            if normalized_names[col] in id_names
        ]

        named_target_columns = [
            col
            for col in df.columns
            if normalized_names[col] in target_names
        ]

        # Highest priority: a saved pandas CSV index.
        if len(unnamed_columns) == 1:
            removed_column = unnamed_columns[0]

        # Next priority: a clearly named genome/sample identifier.
        elif len(named_id_columns) == 1:
            removed_column = named_id_columns[0]

        # This should not normally occur, but handle a target column
        # safely if the CSV accidentally contains one.
        elif len(named_target_columns) == 1:
            removed_column = named_target_columns[0]

        else:
            # Final safe fallback:
            # a true genome-ID column should contain one unique value
            # per genome, unlike a binary gene feature.
            unique_columns = [
                col
                for col in df.columns
                if df[col].nunique(dropna=False) == len(df)
            ]

            if len(unique_columns) == 1:
                removed_column = unique_columns[0]

            else:
                raise ValueError(
                    "The feature CSV has 5,607 columns, but the "
                    "single non-feature column could not be identified "
                    "safely.\n"
                    f"Unnamed candidates: {unnamed_columns}\n"
                    f"Named ID candidates: {named_id_columns}\n"
                    f"Named target candidates: {named_target_columns}\n"
                    f"Fully unique candidates: {unique_columns[:20]}"
                )

        removed_name = str(
            removed_column
        ).strip().lower()

        if (
            removed_name in id_names
            or removed_name.startswith("unnamed")
            or df[removed_column].nunique(
                dropna=False
            ) == len(df)
        ):
            sample_ids = (
                df[removed_column]
                .astype(str)
                .to_numpy()
            )

        print(
            "Dropping non-feature column:",
            removed_column,
        )

        df = df.drop(
            columns=[removed_column]
        )

    else:
        raise ValueError(
            "Unexpected raw feature-matrix width.\n"
            f"Expected 5,606 or 5,607 columns, found {df.shape[1]}."
        )

    # ------------------------------------------------------------
    # Validate final dimensions.
    # ------------------------------------------------------------

    if df.shape != (
        EXPECTED_N,
        EXPECTED_FEATURES,
    ):
        raise ValueError(
            "Feature matrix has the wrong final shape.\n"
            f"Expected {(EXPECTED_N, EXPECTED_FEATURES)}, "
            f"found {df.shape}."
        )

    # ------------------------------------------------------------
    # Convert every gene feature to numeric.
    # ------------------------------------------------------------

    numeric_df = df.apply(
        pd.to_numeric,
        errors="coerce",
    )

    bad_columns = numeric_df.columns[
        numeric_df.isna().any(axis=0)
    ].tolist()

    if bad_columns:
        raise ValueError(
            "Some proposed feature columns contain text or missing "
            "values.\n"
            f"Example problematic columns: {bad_columns[:20]}"
        )

    X = numeric_df.to_numpy(
        dtype=np.float32,
    )

    if not np.isfinite(X).all():
        raise ValueError(
            "Feature matrix contains NaN or infinite values."
        )

    # This is a bag-of-genes presence/absence matrix.
    unique_values = np.unique(X)

    if not set(
        unique_values.tolist()
    ).issubset({0.0, 1.0}):
        raise ValueError(
            "The feature matrix is not binary.\n"
            f"Values found: {unique_values[:20]}"
        )

    if sample_ids is None:
        sample_ids = np.arange(
            EXPECTED_N
        ).astype(str)

        id_source = "generated sequential row indices"

    else:
        id_source = (
            f"removed column '{removed_column}'"
        )

    validation_info = {
        "raw_shape": [
            int(EXPECTED_N),
            int(
                EXPECTED_FEATURES
                + (
                    1
                    if removed_column is not None
                    else 0
                )
            ),
        ],
        "final_shape": [
            int(X.shape[0]),
            int(X.shape[1]),
        ],
        "removed_column": (
            None
            if removed_column is None
            else str(removed_column)
        ),
        "sample_id_source": id_source,
        "feature_values": [
            float(value)
            for value in unique_values
        ],
    }

    return (
        X,
        np.asarray(sample_ids),
        validation_info,
    )


X_matrix, X_sample_ids, X_validation = (
    load_feature_matrix(X_PATH)
)

In [ ]:
# ================================================================
# 5. LOAD AND VALIDATE LABELS
# ================================================================

def load_label_file(path):
    """
    Load the verified RF true-label file.

    Supports:
        0 = susceptible
        1 = resistant

    It also recognizes the older project encoding:
        1 = resistant
        2 = susceptible
    """

    label_df = pd.read_csv(path)

    # Handle a possible headerless single-column file.
    if len(label_df) == EXPECTED_N - 1:
        alternative = pd.read_csv(
            path,
            header=None,
        )

        if len(alternative) == EXPECTED_N:
            label_df = alternative

    print("\n" + "=" * 72)
    print("RAW LABEL-FILE INSPECTION")
    print("=" * 72)

    print(f"Raw y table shape: {label_df.shape}")
    print(
        "Label-file columns:",
        [
            str(col)
            for col in label_df.columns
        ],
    )

    normalized_names = {
        col: str(col).strip().lower()
        for col in label_df.columns
    }

    preferred_label_names = [
        "true_label",
        "true_labels",
        "label",
        "labels",
        "target",
        "y",
        "phenotype",
        "resistance_status",
        "resistance",
        "prediction",
        "predicted_label",
        "0",
    ]

    label_column = None

    # ------------------------------------------------------------
    # Prefer a clearly named label column.
    # ------------------------------------------------------------

    for preferred_name in preferred_label_names:
        matching = [
            col
            for col in label_df.columns
            if normalized_names[col] == preferred_name
        ]

        if len(matching) == 1:
            label_column = matching[0]
            break

    # ------------------------------------------------------------
    # Otherwise select the only column that contains binary labels.
    # ------------------------------------------------------------

    if label_column is None:
        numeric_binary_candidates = []

        for col in label_df.columns:
            values = pd.to_numeric(
                label_df[col],
                errors="coerce",
            )

            if values.notna().all():
                unique_values = set(
                    values.unique().tolist()
                )

                if unique_values.issubset(
                    {0, 1}
                ):
                    numeric_binary_candidates.append(
                        col
                    )

        if len(
            numeric_binary_candidates
        ) == 1:
            label_column = (
                numeric_binary_candidates[0]
            )

    # ------------------------------------------------------------
    # If only one non-index column exists, use it.
    # ------------------------------------------------------------

    if label_column is None:
        non_index_columns = [
            col
            for col in label_df.columns
            if not normalized_names[col].startswith(
                "unnamed"
            )
        ]

        if len(non_index_columns) == 1:
            label_column = non_index_columns[0]

    if label_column is None:
        raise ValueError(
            "Could not safely identify the label column.\n"
            f"Columns found: {list(label_df.columns)}"
        )

    print(
        "Selected label column:",
        label_column,
    )

    raw_labels = label_df[
        label_column
    ].copy()

    numeric_labels = pd.to_numeric(
        raw_labels,
        errors="coerce",
    )

    # ------------------------------------------------------------
    # Numeric encoding.
    # ------------------------------------------------------------

    if numeric_labels.notna().all():

        unique_values = set(
            numeric_labels.unique().tolist()
        )

        if unique_values.issubset(
            {0, 1}
        ):
            y = numeric_labels.astype(
                np.int64
            ).to_numpy()

            label_mapping = {
                "0": "susceptible",
                "1": "resistant",
            }

        elif unique_values.issubset(
            {1, 2}
        ):
            y = numeric_labels.map(
                {
                    1: 1,
                    2: 0,
                }
            ).astype(
                np.int64
            ).to_numpy()

            label_mapping = {
                "1": "resistant",
                "2": "susceptible",
            }

        else:
            raise ValueError(
                "Unrecognized numeric label values.\n"
                f"Values found: {sorted(unique_values)}"
            )

    # ------------------------------------------------------------
    # Text encoding.
    # ------------------------------------------------------------

    else:
        text_labels = (
            raw_labels
            .astype(str)
            .str.strip()
            .str.lower()
        )

        y_series = pd.Series(
            np.full(
                len(text_labels),
                np.nan,
            )
        )

        susceptible_mask = (
            text_labels.str.contains(
                "suscept",
                regex=False,
            )
            | text_labels.isin(
                {"s", "0"}
            )
        )

        resistant_mask = (
            text_labels.str.contains(
                "resist",
                regex=False,
            )
            | text_labels.isin(
                {"r", "1"}
            )
        )

        y_series.loc[
            susceptible_mask
        ] = 0

        y_series.loc[
            resistant_mask
        ] = 1

        if y_series.isna().any():
            unknown_values = sorted(
                text_labels[
                    y_series.isna()
                ].unique().tolist()
            )

            raise ValueError(
                "Unrecognized text labels were found.\n"
                f"Values: {unknown_values[:20]}"
            )

        y = y_series.astype(
            np.int64
        ).to_numpy()

        label_mapping = {
            "susceptible text": 0,
            "resistant text": 1,
        }

    # ------------------------------------------------------------
    # Locate a genome/sample ID column when available.
    # ------------------------------------------------------------

    id_names = {
        "genome_id",
        "genome",
        "genome id",
        "sample_id",
        "sample",
        "sample id",
        "isolate_id",
        "isolate",
        "isolate id",
        "id",
        "index",
        "row_index",
        "row index",
    }

    label_ids = None
    id_column = None

    for col in label_df.columns:

        if col == label_column:
            continue

        normalized = normalized_names[col]

        if (
            normalized in id_names
            or normalized.startswith("unnamed")
        ):
            if label_df[col].nunique(
                dropna=False
            ) == len(label_df):
                id_column = col
                label_ids = (
                    label_df[col]
                    .astype(str)
                    .to_numpy()
                )
                break

    validation_info = {
        "raw_shape": [
            int(label_df.shape[0]),
            int(label_df.shape[1]),
        ],
        "label_column": str(label_column),
        "id_column": (
            None
            if id_column is None
            else str(id_column)
        ),
        "label_mapping": label_mapping,
    }

    return (
        y,
        label_ids,
        validation_info,
    )


y_values, y_sample_ids, y_validation = (
    load_label_file(Y_PATH)
)

In [ ]:
# ================================================================
# 6. ALIGN X AND y
# ================================================================

if len(y_values) != EXPECTED_N:
    raise ValueError(
        "Label count is incorrect.\n"
        f"Expected {EXPECTED_N}, found {len(y_values)}."
    )

if len(X_matrix) != EXPECTED_N:
    raise ValueError(
        "Feature row count is incorrect.\n"
        f"Expected {EXPECTED_N}, found {len(X_matrix)}."
    )

alignment_method = "existing row order"

# Use explicit identifiers when both files provide them.
if (
    X_sample_ids is not None
    and y_sample_ids is not None
):
    x_ids = pd.Series(
        X_sample_ids
    ).astype(str).str.strip()

    y_ids = pd.Series(
        y_sample_ids
    ).astype(str).str.strip()

    if x_ids.duplicated().any():
        raise ValueError(
            "Duplicate IDs were found in the feature matrix."
        )

    if y_ids.duplicated().any():
        raise ValueError(
            "Duplicate IDs were found in the label file."
        )

    if set(x_ids) == set(y_ids):
        y_lookup = pd.Series(
            y_values,
            index=y_ids,
        )

        y_values = (
            y_lookup
            .loc[x_ids]
            .to_numpy(
                dtype=np.int64
            )
        )

        alignment_method = (
            "labels reordered to match explicit sample IDs"
        )

        print(
            "\nPASS: X and y were aligned using "
            "explicit sample identifiers."
        )

    else:
        raise ValueError(
            "Both X and y contain identifiers, but their ID sets "
            "do not match. Training was stopped to prevent row "
            "misalignment."
        )

else:
    print(
        "\nNote: explicit IDs were not present in both files. "
        "The verified saved row order will be used."
    )

In [ ]:
# ================================================================
# 7. FINAL DATASET VALIDATION
# ================================================================

class_counts = (
    pd.Series(y_values)
    .value_counts()
    .sort_index()
)

susceptible_count = int(
    class_counts.get(0, 0)
)

resistant_count = int(
    class_counts.get(1, 0)
)

print("\n" + "=" * 72)
print("FINAL DATASET VALIDATION")
print("=" * 72)

print(f"X shape: {X_matrix.shape}")
print(f"y shape: {y_values.shape}")
print(
    f"Susceptible (0): {susceptible_count}"
)
print(
    f"Resistant   (1): {resistant_count}"
)
print(
    f"Alignment method: {alignment_method}"
)

assert X_matrix.shape == (
    EXPECTED_N,
    EXPECTED_FEATURES,
), (
    f"Expected X shape "
    f"{(EXPECTED_N, EXPECTED_FEATURES)}, "
    f"found {X_matrix.shape}"
)

assert y_values.shape == (
    EXPECTED_N,
), (
    f"Expected y shape {(EXPECTED_N,)}, "
    f"found {y_values.shape}"
)

assert susceptible_count == (
    EXPECTED_SUSCEPTIBLE
), (
    f"Expected {EXPECTED_SUSCEPTIBLE} susceptible "
    f"genomes, found {susceptible_count}"
)

assert resistant_count == (
    EXPECTED_RESISTANT
), (
    f"Expected {EXPECTED_RESISTANT} resistant "
    f"genomes, found {resistant_count}"
)

assert set(
    np.unique(y_values).tolist()
) == {0, 1}

print(
    "\nPASS: final internal matrix is exactly "
    "2,448 × 5,606."
)

print(
    "PASS: labels are binary with the expected "
    "1,815 susceptible and 633 resistant genomes."
)

In [ ]:
# ================================================================
# 8. MODEL DEFINITIONS
# ================================================================

def create_random_forest(seed):
    return RandomForestClassifier(
        n_estimators=100,
        criterion="gini",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )


def create_logistic_regression(seed):
    return LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        class_weight=None,
        random_state=seed,
    )


MODEL_FACTORIES = {
    "Random Forest": create_random_forest,
    "Logistic Regression": (
        create_logistic_regression
    ),
}

In [ ]:
# ================================================================
# 9. REPEATED FIVE-SEED, 10-FOLD EVALUATION
# ================================================================

fold_rows = []
prediction_rows = []

total_fits = (
    len(MODEL_FACTORIES)
    * len(SEEDS)
    * N_SPLITS
)

completed_fits = 0
overall_start = time.perf_counter()

for model_name, model_factory in (
    MODEL_FACTORIES.items()
):

    print("\n" + "=" * 72)
    print(f"STARTING MODEL: {model_name}")
    print("=" * 72)

    for seed in SEEDS:

        print(f"\nSeed {seed}")

        cv = StratifiedKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed,
        )

        for fold_number, (
            train_idx,
            test_idx,
        ) in enumerate(
            cv.split(
                X_matrix,
                y_values,
            ),
            start=1,
        ):
            fold_start = time.perf_counter()

            model = model_factory(seed)

            convergence_warning = False
            warning_messages = []

            with warnings.catch_warnings(
                record=True
            ) as caught_warnings:

                warnings.simplefilter(
                    "always"
                )

                model.fit(
                    X_matrix[train_idx],
                    y_values[train_idx],
                )

                for warning in caught_warnings:

                    warning_messages.append(
                        str(warning.message)
                    )

                    if issubclass(
                        warning.category,
                        ConvergenceWarning,
                    ):
                        convergence_warning = True

            probabilities = (
                model.predict_proba(
                    X_matrix[test_idx]
                )[:, 1]
            )

            predictions = (
                probabilities >= THRESHOLD
            ).astype(np.int64)

            y_test = y_values[test_idx]

            tn, fp, fn, tp = confusion_matrix(
                y_test,
                predictions,
                labels=[0, 1],
            ).ravel()

            accuracy = accuracy_score(
                y_test,
                predictions,
            )

            balanced_accuracy = (
                balanced_accuracy_score(
                    y_test,
                    predictions,
                )
            )

            roc_auc = roc_auc_score(
                y_test,
                probabilities,
            )

            precision = precision_score(
                y_test,
                predictions,
                zero_division=0,
            )

            recall = recall_score(
                y_test,
                predictions,
                zero_division=0,
            )

            specificity = (
                tn / (tn + fp)
                if (tn + fp) > 0
                else np.nan
            )

            f1 = f1_score(
                y_test,
                predictions,
                zero_division=0,
            )

            runtime_seconds = (
                time.perf_counter()
                - fold_start
            )

            n_iterations = np.nan

            if hasattr(model, "n_iter_"):
                try:
                    n_iterations = int(
                        np.max(model.n_iter_)
                    )
                except Exception:
                    pass

            fold_rows.append(
                {
                    "Model": model_name,
                    "Seed": seed,
                    "Fold": fold_number,
                    "N train": len(train_idx),
                    "N test": len(test_idx),
                    "TN": int(tn),
                    "FP": int(fp),
                    "FN": int(fn),
                    "TP": int(tp),
                    "Accuracy": accuracy,
                    "Balanced accuracy": (
                        balanced_accuracy
                    ),
                    "ROC-AUC": roc_auc,
                    "Precision": precision,
                    "Recall/Sensitivity": recall,
                    "Specificity": specificity,
                    "F1": f1,
                    "Threshold": THRESHOLD,
                    "Runtime seconds": (
                        runtime_seconds
                    ),
                    "Iterations": n_iterations,
                    "Convergence warning": (
                        convergence_warning
                    ),
                    "Warning messages": (
                        " | ".join(
                            warning_messages
                        )
                    ),
                }
            )

            for local_position, sample_index in (
                enumerate(test_idx)
            ):
                prediction_rows.append(
                    {
                        "Model": model_name,
                        "Seed": seed,
                        "Fold": fold_number,
                        "Row index": int(
                            sample_index
                        ),
                        "Sample ID": str(
                            X_sample_ids[
                                sample_index
                            ]
                        ),
                        "True label": int(
                            y_values[
                                sample_index
                            ]
                        ),
                        (
                            "Predicted probability "
                            "resistant"
                        ): float(
                            probabilities[
                                local_position
                            ]
                        ),
                        "Predicted label": int(
                            predictions[
                                local_position
                            ]
                        ),
                        "Threshold": THRESHOLD,
                    }
                )

            completed_fits += 1

            elapsed_minutes = (
                time.perf_counter()
                - overall_start
            ) / 60

            average_fit_minutes = (
                elapsed_minutes
                / completed_fits
            )

            estimated_remaining = (
                total_fits
                - completed_fits
            ) * average_fit_minutes

            print(
                f"  Fold {fold_number:02d}/"
                f"{N_SPLITS} | "
                f"Accuracy={accuracy:.4f} | "
                f"ROC-AUC={roc_auc:.4f} | "
                f"F1={f1:.4f} | "
                f"Runtime={runtime_seconds:.1f}s | "
                f"Approx. remaining="
                f"{estimated_remaining:.1f} min"
            )

        # Save a checkpoint after every completed seed.
        pd.DataFrame(
            fold_rows
        ).to_csv(
            RUN_DIR
            / "CHECKPOINT_fold_results.csv",
            index=False,
        )

        pd.DataFrame(
            prediction_rows
        ).to_csv(
            RUN_DIR
            / "CHECKPOINT_predictions.csv",
            index=False,
        )

In [ ]:
# ================================================================
# 10. BUILD RESULT TABLES
# ================================================================

fold_df = pd.DataFrame(
    fold_rows
)

predictions_df = pd.DataFrame(
    prediction_rows
)

expected_fold_rows = (
    len(MODEL_FACTORIES)
    * len(SEEDS)
    * N_SPLITS
)

expected_prediction_rows = (
    len(MODEL_FACTORIES)
    * len(SEEDS)
    * EXPECTED_N
)

assert len(fold_df) == expected_fold_rows, (
    f"Expected {expected_fold_rows} fold rows, "
    f"found {len(fold_df)}."
)

assert (
    len(predictions_df)
    == expected_prediction_rows
), (
    f"Expected {expected_prediction_rows} "
    f"prediction rows, "
    f"found {len(predictions_df)}."
)

metric_columns = [
    "Accuracy",
    "Balanced accuracy",
    "ROC-AUC",
    "Precision",
    "Recall/Sensitivity",
    "Specificity",
    "F1",
]

In [ ]:
# ================================================================
# 11. SUMMARY BY SEED
# ================================================================

seed_summary_rows = []

for (
    model_name,
    seed,
), group in fold_df.groupby(
    [
        "Model",
        "Seed",
    ],
    sort=False,
):

    summary_row = {
        "Model": model_name,
        "Seed": int(seed),
        "Folds completed": len(group),
        "Mean runtime seconds per fold": (
            group[
                "Runtime seconds"
            ].mean()
        ),
        "Total TN": int(
            group["TN"].sum()
        ),
        "Total FP": int(
            group["FP"].sum()
        ),
        "Total FN": int(
            group["FN"].sum()
        ),
        "Total TP": int(
            group["TP"].sum()
        ),
        "Convergence warnings": int(
            group[
                "Convergence warning"
            ].sum()
        ),
    }

    for metric in metric_columns:
        summary_row[
            f"{metric} mean"
        ] = group[metric].mean()

        summary_row[
            f"{metric} SD"
        ] = group[metric].std(
            ddof=1
        )

    seed_summary_rows.append(
        summary_row
    )

summary_by_seed_df = pd.DataFrame(
    seed_summary_rows
)

In [ ]:
# ================================================================
# 12. OVERALL FIVE-SEED SUMMARY
# ================================================================

overall_summary_rows = []

for model_name, group in fold_df.groupby(
    "Model",
    sort=False,
):

    model_seed_summary = (
        summary_by_seed_df[
            summary_by_seed_df[
                "Model"
            ] == model_name
        ]
    )

    summary_row = {
        "Model": model_name,
        "Feature representation": (
            "Prokka bag-of-genes"
        ),
        "Evaluation context": (
            "Internal repeated 10-fold "
            "stratified cross-validation"
        ),
        "Seeds": ",".join(
            str(seed)
            for seed in SEEDS
        ),
        "Number of seeds": len(SEEDS),
        "Folds per seed": N_SPLITS,
        "Total fold evaluations": (
            len(group)
        ),
        "N": EXPECTED_N,
        "Susceptible": (
            EXPECTED_SUSCEPTIBLE
        ),
        "Resistant": (
            EXPECTED_RESISTANT
        ),
        "Threshold": THRESHOLD,
        "Total TN": int(
            group["TN"].sum()
        ),
        "Total FP": int(
            group["FP"].sum()
        ),
        "Total FN": int(
            group["FN"].sum()
        ),
        "Total TP": int(
            group["TP"].sum()
        ),
        "Mean runtime seconds per fold": (
            group[
                "Runtime seconds"
            ].mean()
        ),
        "Convergence warnings": int(
            group[
                "Convergence warning"
            ].sum()
        ),
    }

    for metric in metric_columns:

        # This matches the reporting used for the other
        # five-seed models: mean and SD across all 50 folds.
        summary_row[
            f"{metric} mean"
        ] = group[metric].mean()

        summary_row[
            f"{metric} SD"
        ] = group[metric].std(
            ddof=1
        )

        # Also save variation among the five seed-specific means.
        summary_row[
            f"{metric} seed-mean SD"
        ] = model_seed_summary[
            f"{metric} mean"
        ].std(
            ddof=1
        )

    overall_summary_rows.append(
        summary_row
    )

overall_summary_df = pd.DataFrame(
    overall_summary_rows
)

In [ ]:
# ================================================================
# 13. COMPACT REVIEWER TABLE
# ================================================================

compact_rows = []

for _, row in (
    overall_summary_df.iterrows()
):

    compact_rows.append(
        {
            "Model": row["Model"],
            "Seeds": row["Seeds"],
            "Folds": int(
                row[
                    "Total fold evaluations"
                ]
            ),
            "N": int(row["N"]),
            "Accuracy mean ± SD": (
                f"{row['Accuracy mean']:.4f} "
                f"± {row['Accuracy SD']:.4f}"
            ),
            (
                "Balanced accuracy "
                "mean ± SD"
            ): (
                f"{row['Balanced accuracy mean']:.4f} "
                f"± "
                f"{row['Balanced accuracy SD']:.4f}"
            ),
            "ROC-AUC mean ± SD": (
                f"{row['ROC-AUC mean']:.4f} "
                f"± {row['ROC-AUC SD']:.4f}"
            ),
            "Precision mean ± SD": (
                f"{row['Precision mean']:.4f} "
                f"± {row['Precision SD']:.4f}"
            ),
            (
                "Recall/Sensitivity "
                "mean ± SD"
            ): (
                f"{row['Recall/Sensitivity mean']:.4f} "
                f"± "
                f"{row['Recall/Sensitivity SD']:.4f}"
            ),
            "Specificity mean ± SD": (
                f"{row['Specificity mean']:.4f} "
                f"± {row['Specificity SD']:.4f}"
            ),
            "F1 mean ± SD": (
                f"{row['F1 mean']:.4f} "
                f"± {row['F1 SD']:.4f}"
            ),
        }
    )

compact_df = pd.DataFrame(
    compact_rows
)

In [ ]:
# ================================================================
# 14. VERIFY COMPLETE OUT-OF-FOLD COVERAGE
# ================================================================

coverage_rows = []

for (
    model_name,
    seed,
), group in predictions_df.groupby(
    [
        "Model",
        "Seed",
    ]
):

    unique_rows = group[
        "Row index"
    ].nunique()

    duplicate_rows = (
        len(group)
        - unique_rows
    )

    coverage_rows.append(
        {
            "Model": model_name,
            "Seed": int(seed),
            "Prediction rows": len(group),
            "Unique genome rows": (
                unique_rows
            ),
            "Duplicate genome rows": (
                duplicate_rows
            ),
            "Complete coverage": (
                len(group) == EXPECTED_N
                and unique_rows
                == EXPECTED_N
                and duplicate_rows == 0
            ),
        }
    )

coverage_df = pd.DataFrame(
    coverage_rows
)

assert coverage_df[
    "Complete coverage"
].all(), (
    "At least one model/seed did not produce "
    "exactly one held-out prediction per genome."
)

In [ ]:
# ================================================================
# 15. SAVE CSV OUTPUTS
# ================================================================

fold_path = (
    RUN_DIR
    / "rf_lr_10fold_fold_level_results_5seeds.csv"
)

predictions_path = (
    RUN_DIR
    / "rf_lr_10fold_predictions_5seeds.csv"
)

seed_summary_path = (
    RUN_DIR
    / "rf_lr_10fold_summary_by_seed.csv"
)

overall_summary_path = (
    RUN_DIR
    / "rf_lr_10fold_summary_5seeds.csv"
)

compact_path = (
    RUN_DIR
    / "rf_lr_10fold_compact_reviewer_table.csv"
)

coverage_path = (
    RUN_DIR
    / "rf_lr_10fold_prediction_coverage_check.csv"
)

fold_df.to_csv(
    fold_path,
    index=False,
)

predictions_df.to_csv(
    predictions_path,
    index=False,
)

summary_by_seed_df.to_csv(
    seed_summary_path,
    index=False,
)

overall_summary_df.to_csv(
    overall_summary_path,
    index=False,
)

compact_df.to_csv(
    compact_path,
    index=False,
)

coverage_df.to_csv(
    coverage_path,
    index=False,
)

In [ ]:
# ================================================================
# 16. SAVE ACCURACY-BY-SEED FIGURE
# ================================================================

accuracy_plot_path = (
    RUN_DIR
    / "rf_lr_accuracy_by_seed.png"
)

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

model_names = list(
    MODEL_FACTORIES.keys()
)

x_positions = np.arange(
    len(SEEDS)
)

bar_width = 0.36

for model_position, model_name in enumerate(
    model_names
):

    model_data = (
        summary_by_seed_df[
            summary_by_seed_df[
                "Model"
            ] == model_name
        ]
        .set_index("Seed")
        .loc[SEEDS]
    )

    if model_position == 0:
        offsets = (
            x_positions
            - bar_width / 2
        )
    else:
        offsets = (
            x_positions
            + bar_width / 2
        )

    ax.bar(
        offsets,
        model_data[
            "Accuracy mean"
        ].to_numpy(),
        bar_width,
        yerr=model_data[
            "Accuracy SD"
        ].to_numpy(),
        capsize=4,
        label=model_name,
    )

ax.set_xlabel(
    "Random seed"
)

ax.set_ylabel(
    "Mean 10-fold accuracy"
)

ax.set_title(
    "Random Forest and Logistic Regression "
    "Accuracy Across Seeds"
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    [
        str(seed)
        for seed in SEEDS
    ]
)

ax.set_ylim(
    0.80,
    1.00,
)

ax.legend()

fig.tight_layout()

fig.savefig(
    accuracy_plot_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)

In [ ]:
# ================================================================
# 17. SAVE RUN METADATA
# ================================================================

total_runtime_minutes = (
    time.perf_counter()
    - overall_start
) / 60

metadata = {
    "run_tag": RUN_TAG,
    "generated_at": (
        datetime.now().isoformat()
    ),
    "runtime_minutes": (
        total_runtime_minutes
    ),
    "run_directory": str(
        RUN_DIR
    ),
    "X_source": str(
        X_PATH
    ),
    "y_source": str(
        Y_PATH
    ),
    "X_sha256": X_SHA256,
    "y_sha256": Y_SHA256,
    "X_validation": (
        X_validation
    ),
    "y_validation": (
        y_validation
    ),
    "alignment_method": (
        alignment_method
    ),
    "N": EXPECTED_N,
    "feature_count": (
        EXPECTED_FEATURES
    ),
    "susceptible": (
        EXPECTED_SUSCEPTIBLE
    ),
    "resistant": (
        EXPECTED_RESISTANT
    ),
    "seeds": SEEDS,
    "folds_per_seed": (
        N_SPLITS
    ),
    "threshold": THRESHOLD,
    "random_forest_hyperparameters": {
        "n_estimators": 100,
        "criterion": "gini",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "bootstrap": True,
        "class_weight": "balanced",
        "random_state": (
            "current evaluation seed"
        ),
        "n_jobs": -1,
    },
    "logistic_regression_hyperparameters": {
        "penalty": "l2",
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 1000,
        "class_weight": None,
        "random_state": (
            "current evaluation seed; "
            "fold composition changes by seed"
        ),
    },
    "aggregation": {
        "main_mean": (
            "mean across all 50 held-out folds"
        ),
        "main_SD": (
            "sample standard deviation across all "
            "50 held-out folds; ddof=1"
        ),
        "seed_mean_SD": (
            "sample standard deviation across the "
            "five seed-specific means; ddof=1"
        ),
    },
    "software": {
        "python": (
            platform.python_version()
        ),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": (
            sklearn.__version__
        ),
    },
    "master_table_modified": False,
}

metadata_path = (
    RUN_DIR
    / "rf_lr_seed_robustness_run_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        metadata,
        handle,
        indent=2,
    )

In [ ]:
# ================================================================
# 18. SAVE README
# ================================================================

readme_path = (
    RUN_DIR
    / "README_rf_lr_seed_robustness.txt"
)

with open(
    readme_path,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        "Random Forest and Logistic Regression "
        "Five-Seed Robustness\n"
    )

    handle.write(
        "=========================================="
        "================\n\n"
    )

    handle.write(
        f"Generated: "
        f"{datetime.now().isoformat()}\n"
    )

    handle.write(
        f"Runtime minutes: "
        f"{total_runtime_minutes:.2f}\n\n"
    )

    handle.write(
        "Dataset:\n"
    )

    handle.write(
        f"- N = {EXPECTED_N}\n"
    )

    handle.write(
        f"- Features = "
        f"{EXPECTED_FEATURES}\n"
    )

    handle.write(
        f"- Susceptible = "
        f"{EXPECTED_SUSCEPTIBLE}\n"
    )

    handle.write(
        f"- Resistant = "
        f"{EXPECTED_RESISTANT}\n"
    )

    handle.write(
        f"- Alignment = "
        f"{alignment_method}\n\n"
    )

    handle.write(
        "Evaluation:\n"
    )

    handle.write(
        f"- Seeds = {SEEDS}\n"
    )

    handle.write(
        f"- Folds per seed = "
        f"{N_SPLITS}\n"
    )

    handle.write(
        "- Total held-out folds per model = 50\n"
    )

    handle.write(
        f"- Threshold = "
        f"{THRESHOLD}\n\n"
    )

    handle.write(
        "Models:\n"
    )

    handle.write(
        "- Random Forest: 100 trees, Gini, "
        "max_depth=None, class_weight=balanced\n"
    )

    handle.write(
        "- Logistic Regression: L2, C=1.0, "
        "LBFGS, max_iter=1000\n\n"
    )

    handle.write(
        "Important:\n"
    )

    handle.write(
        "- No hyperparameter tuning was performed.\n"
    )

    handle.write(
        "- The master performance table was not modified.\n"
    )

    handle.write(
        "- Results should be reviewed before manuscript "
        "integration.\n"
    )

In [ ]:
# ================================================================
# 19. CREATE LATEST COPIES
# ================================================================

latest_files = {
    fold_path: (
        BASE_OUT_DIR
        / "rf_lr_10fold_fold_level_results_5seeds_LATEST.csv"
    ),
    predictions_path: (
        BASE_OUT_DIR
        / "rf_lr_10fold_predictions_5seeds_LATEST.csv"
    ),
    seed_summary_path: (
        BASE_OUT_DIR
        / "rf_lr_10fold_summary_by_seed_LATEST.csv"
    ),
    overall_summary_path: (
        BASE_OUT_DIR
        / "rf_lr_10fold_summary_5seeds_LATEST.csv"
    ),
    compact_path: (
        BASE_OUT_DIR
        / "rf_lr_10fold_compact_reviewer_table_LATEST.csv"
    ),
    coverage_path: (
        BASE_OUT_DIR
        / "rf_lr_10fold_prediction_coverage_check_LATEST.csv"
    ),
    accuracy_plot_path: (
        BASE_OUT_DIR
        / "rf_lr_accuracy_by_seed_LATEST.png"
    ),
    metadata_path: (
        BASE_OUT_DIR
        / "rf_lr_seed_robustness_run_metadata_LATEST.json"
    ),
    readme_path: (
        BASE_OUT_DIR
        / "README_rf_lr_seed_robustness_LATEST.txt"
    ),
}

for source_path, destination_path in (
    latest_files.items()
):
    shutil.copy2(
        source_path,
        destination_path,
    )

In [ ]:
# ================================================================
# 20. FINAL OUTPUT
# ================================================================

print("\n" + "=" * 72)
print("FIVE-SEED RF + LR ANALYSIS COMPLETE")
print("=" * 72)

print(
    f"\nTotal runtime: "
    f"{total_runtime_minutes:.2f} minutes"
)

display_columns = [
    "Model",
    "Accuracy mean",
    "Accuracy SD",
    "Accuracy seed-mean SD",
    "Balanced accuracy mean",
    "Balanced accuracy SD",
    "ROC-AUC mean",
    "ROC-AUC SD",
    "Precision mean",
    "Precision SD",
    "Recall/Sensitivity mean",
    "Recall/Sensitivity SD",
    "Specificity mean",
    "Specificity SD",
    "F1 mean",
    "F1 SD",
]

print(
    "\nOVERALL FIVE-SEED SUMMARY:"
)

print(
    overall_summary_df[
        display_columns
    ]
    .round(4)
    .to_string(
        index=False
    )
)

seed_display_columns = [
    "Model",
    "Seed",
    "Accuracy mean",
    "Accuracy SD",
    "ROC-AUC mean",
    "ROC-AUC SD",
    "F1 mean",
    "F1 SD",
    "Convergence warnings",
]

print(
    "\nSUMMARY BY SEED:"
)

print(
    summary_by_seed_df[
        seed_display_columns
    ]
    .round(4)
    .to_string(
        index=False
    )
)

print(
    "\nOUT-OF-FOLD COVERAGE CHECK:"
)

print(
    coverage_df.to_string(
        index=False
    )
)

print(
    "\nSaved timestamped run directory:"
)

print(
    RUN_DIR
)

print(
    "\nLatest summary file:"
)

print(
    BASE_OUT_DIR
    / "rf_lr_10fold_summary_5seeds_LATEST.csv"
)

print(
    "\nLatest compact reviewer table:"
)

print(
    BASE_OUT_DIR
    / "rf_lr_10fold_compact_reviewer_table_LATEST.csv"
)

print(
    "\nMaster performance table modified: NO"
)

print(
    "\nReview the results before replacing the current "
    "Random Forest and Logistic Regression rows."
)